# PRE-ANÁLISIS

In [312]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [313]:
datos = pd.read_csv("diabetic_data.csv")

In [314]:
# display(datos.head()) # primeras 5
# display(datos.dtypes) # tipos
# datos.describe(include='all') # analiza media, std etc
# display(datos.isnull().any()) # valores null
display(datos.info())
# datos.describe()
# display(datos)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

None

In [315]:
datos["readmitted"] = datos["readmitted"].astype("category")
datos["readmitted"] = datos["readmitted"].map({
    "<30": 1,
    ">30": 0,
    "NO": 0
})

In [316]:
# Reemplazar los valores "?" por NaN (not a number)
datos = datos.replace("?", np.nan)
datos = datos.replace("None", np.nan)


In [317]:
# Quitar IDs (innecesarios para el análisis) y columnas con muchos valores faltantes
datos = datos.drop(columns=['encounter_id', 'patient_nbr'])
datos = datos.drop(columns=['weight'])
datos = datos.drop(columns=['max_glu_serum'])
datos = datos.drop(columns=['A1Cresult'])
datos = datos.drop(columns=['medical_specialty'])
datos = datos.drop(columns=['payer_code'])


Codificar variables categóricas

In [318]:
datos["race"].value_counts() # V. Categórica sin orden, usamos one-hot encoding

race
Caucasian          76099
AfricanAmerican    19210
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [319]:
datos = pd.get_dummies(datos, columns=['race'], drop_first=True)

In [320]:
datos["gender"].value_counts() # V. Categórica sin orden, usamos one-hot encoding

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

In [321]:
datos = pd.get_dummies(datos, columns=['gender'], drop_first=True)

In [322]:
datos["age"].value_counts() # V. Categórica con orden, mapearé con valor medio de cada rango

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

In [323]:
datos["age"] = datos["age"].map({
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
})

In [324]:
display(datos['diag_1'].value_counts().head()) # |
display(datos['diag_2'].value_counts().head()) # | Como tiene tantas categorías, se agruparán por sistemas.
display(datos['diag_3'].value_counts().head()) # |

diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
Name: count, dtype: int64

diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
Name: count, dtype: int64

diag_3
250    11555
401     8289
276     5175
428     4577
427     3955
Name: count, dtype: int64

In [325]:
def seleccionarDiagnostico(codigo):
    cod = str(codigo).strip()
    try:
        codigoFloat = float(cod)
        if 390 <= codigoFloat <= 459 or codigoFloat == 785:
            return 'Circulatorio'
        elif 460 <= codigoFloat <= 519 or codigoFloat == 786:
            return 'Respiratorio'
        elif 520 <= codigoFloat <= 579 or codigoFloat == 787:
            return 'Digestivo'
        elif 250 <= codigoFloat < 251: # Diabetes es 250.xx
            return 'Diabetes'
        elif 800 <= codigoFloat <= 999:
            return 'Lesiones'
        elif 710 <= codigoFloat <= 739:
            return 'Musculoesquelético'
        elif 580 <= codigoFloat <= 629 or codigoFloat == 788:
            return 'Genitourinario'
        elif 140 <= codigoFloat <= 239:
            return 'Neoplasias'
        else:
            return 'Otros'
    except:
        return "Otros"

for col in ['diag_1', 'diag_2', 'diag_3']:
    datos[col] = datos[col].apply(seleccionarDiagnostico)
    datos = pd.get_dummies(datos, columns=[col], drop_first=True)


In [326]:
datos["metformin"].value_counts() 

metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: count, dtype: int64

In [327]:
datos["metformin"] = datos["metformin"].astype("category")
datos["metformin"] = datos["metformin"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [328]:
datos["repaglinide"].value_counts(normalize="True") 

repaglinide
No        0.984877
Steady    0.013600
Up        0.001081
Down      0.000442
Name: proportion, dtype: float64

In [329]:
datos = datos.drop(columns=['repaglinide'])

In [330]:
datos["nateglinide"].value_counts(normalize="True") 

nateglinide
No        0.993092
Steady    0.006564
Up        0.000236
Down      0.000108
Name: proportion, dtype: float64

In [331]:
datos = datos.drop(columns=['nateglinide'])

In [332]:
datos["chlorpropamide"].value_counts(normalize="True") 

chlorpropamide
No        0.999155
Steady    0.000776
Up        0.000059
Down      0.000010
Name: proportion, dtype: float64

In [333]:
datos = datos.drop(columns=['chlorpropamide'])

In [334]:
datos["glimepiride"].value_counts(normalize="True") 

glimepiride
No        0.948991
Steady    0.045890
Up        0.003213
Down      0.001906
Name: proportion, dtype: float64

In [335]:
datos["glimepiride"] = datos["glimepiride"].astype("category")
datos["glimepiride"] = datos["glimepiride"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [336]:
datos["acetohexamide"].value_counts(normalize="True") 

acetohexamide
No        0.99999
Steady    0.00001
Name: proportion, dtype: float64

In [337]:
datos = datos.drop(columns=['acetohexamide'])

In [338]:
datos["glipizide"].value_counts(normalize="True") 

glipizide
No        0.875341
Steady    0.111589
Up        0.007566
Down      0.005503
Name: proportion, dtype: float64

In [339]:
datos["glipizide"] = datos["glipizide"].astype("category")
datos["glipizide"] = datos["glipizide"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [340]:
datos["glyburide"].value_counts(normalize="True") 

glyburide
No        0.895348
Steady    0.091131
Up        0.007979
Down      0.005542
Name: proportion, dtype: float64

In [341]:
datos["glyburide"] = datos["glyburide"].astype("category")
datos["glyburide"] = datos["glyburide"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [342]:
datos["tolbutamide"].value_counts(normalize="True") 

tolbutamide
No        0.999774
Steady    0.000226
Name: proportion, dtype: float64

In [343]:
datos = datos.drop(columns=['tolbutamide'])

In [344]:
datos["pioglitazone"].value_counts(normalize="True") 

pioglitazone
No        0.927992
Steady    0.068549
Up        0.002299
Down      0.001160
Name: proportion, dtype: float64

In [345]:
datos["pioglitazone"] = datos["pioglitazone"].astype("category")
datos["pioglitazone"] = datos["pioglitazone"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [346]:
datos["rosiglitazone"].value_counts(normalize="True") 

rosiglitazone
No        0.937455
Steady    0.059941
Up        0.001749
Down      0.000855
Name: proportion, dtype: float64

In [347]:
datos["rosiglitazone"] = datos["rosiglitazone"].astype("category")
datos["rosiglitazone"] = datos["rosiglitazone"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [348]:
datos["acarbose"].value_counts(normalize="True") 

acarbose
No        0.996973
Steady    0.002899
Up        0.000098
Down      0.000029
Name: proportion, dtype: float64

In [349]:
datos = datos.drop(columns=['acarbose'])

In [350]:
datos["miglitol"].value_counts(normalize="True") 

miglitol
No        0.999627
Steady    0.000305
Down      0.000049
Up        0.000020
Name: proportion, dtype: float64

In [351]:
datos = datos.drop(columns=['miglitol'])

In [352]:
datos["troglitazone"].value_counts(normalize="True") 

troglitazone
No        0.999971
Steady    0.000029
Name: proportion, dtype: float64

In [353]:
datos = datos.drop(columns=['troglitazone'])

In [354]:
datos["tolazamide"].value_counts(normalize="True") 

tolazamide
No        0.999617
Steady    0.000373
Up        0.000010
Name: proportion, dtype: float64

In [355]:
datos = datos.drop(columns=['tolazamide'])

In [356]:
datos["examide"].value_counts(normalize="True") 

examide
No    1.0
Name: proportion, dtype: float64

In [357]:
datos = datos.drop(columns=['examide'])

In [358]:
datos["citoglipton"].value_counts(normalize="True") 

citoglipton
No    1.0
Name: proportion, dtype: float64

In [359]:
datos = datos.drop(columns=['citoglipton'])

In [360]:
datos["insulin"].value_counts(normalize="True") 

insulin
No        0.465607
Steady    0.303137
Down      0.120060
Up        0.111196
Name: proportion, dtype: float64

In [361]:
datos["insulin"] = datos["insulin"].astype("category")
datos["insulin"] = datos["insulin"].map({
    "No": 0,
    "Steady": 1,
    "Up": 2,
    "Down": 3
})

In [362]:
datos["glyburide-metformin"].value_counts(normalize="True") 

glyburide-metformin
No        0.993063
Steady    0.006800
Up        0.000079
Down      0.000059
Name: proportion, dtype: float64

In [363]:
datos = datos.drop(columns=['glyburide-metformin'])

In [364]:
datos["glipizide-metformin"].value_counts(normalize="True") 

glipizide-metformin
No        0.999872
Steady    0.000128
Name: proportion, dtype: float64

In [365]:
datos = datos.drop(columns=['glipizide-metformin'])

In [366]:
datos["glimepiride-pioglitazone"].value_counts(normalize="True") 

glimepiride-pioglitazone
No        0.99999
Steady    0.00001
Name: proportion, dtype: float64

In [367]:
datos = datos.drop(columns=['glimepiride-pioglitazone'])

In [368]:
datos["metformin-rosiglitazone"].value_counts(normalize="True") 

metformin-rosiglitazone
No        0.99998
Steady    0.00002
Name: proportion, dtype: float64

In [369]:
datos = datos.drop(columns=['metformin-rosiglitazone'])

In [370]:
datos["metformin-pioglitazone"].value_counts(normalize="True") 

metformin-pioglitazone
No        0.99999
Steady    0.00001
Name: proportion, dtype: float64

In [371]:
datos = datos.drop(columns=['metformin-pioglitazone'])

In [372]:
datos["change"].value_counts(normalize="True") 

change
No    0.538048
Ch    0.461952
Name: proportion, dtype: float64

In [373]:
datos["change"] = datos["change"].astype("category")
datos["change"] = datos["change"].map({
    "No": 0,
    "Ch": 1
})

In [374]:
datos["diabetesMed"].value_counts(normalize="True") 

diabetesMed
Yes    0.770031
No     0.229969
Name: proportion, dtype: float64

In [375]:
datos["diabetesMed"] = datos["diabetesMed"].astype("category")
datos["diabetesMed"] = datos["diabetesMed"].map({
    "No": 0,
    "Yes": 1
})

In [380]:
datos["insulin"].value_counts()

insulin
0    47383
1    30849
3    12218
2    11316
Name: count, dtype: int64